# Checking if T4 GPU is used (before ssh)

In [1]:
!nvidia-smi

Tue Oct 13 07:20:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


# Installing dependencies

In [3]:
# Install standard, stable versions compatible with the GPU 
%pip install -q ultralytics roboflow torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.9/373.9 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 140.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.8/96.8 kB 11.5 MB/s eta 0:00:00


In [4]:
from ultralytics import YOLO
from roboflow import Roboflow
from google.colab import files

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [5]:
import os

# Download Vietnamese License Plate Dataset from Roboflow

In [6]:
rf = Roboflow(api_key="RRDysFq49VB9hRc8eXkW")  # Replace with your actual Roboflow API key
# https://app.roboflow.com/alphajack2006-gmail-com/vietnamese-car-license-plate-vlbzz/train
# location of the dataset in Roboflow.
project = rf.workspace("alphajack2006-gmail-com").project("vietnamese-car-license-plate-vlbzz")
version = project.version(1)
dataset = version.download("yolov8") # this one alr works

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Vietnamese-Car-License-Plate--1 in yolov8:: 100%|██████████| 16513/16513 [00:01<00:00, 9241.85it/s] 


Applied Fine Tuning (leveraging a pretrained model to specialise it in finding license plates)

I did not use any optimisation paths like: adding shears or freezing any base layers.


# Train YOLO Models in one big batch

In [6]:
# models_to_train = [
#     {"weights": "yolo26n.pt",  "name": "yolo26n"},
#     {"weights": "yolo26s.pt",  "name": "yolo26s"},
#     {"weights": "yolo11n.pt",  "name": "yolo11n"},
#     {"weights": "yolo11s.pt",  "name": "yolo11s"},
#     {"weights": "yolov10s.pt", "name": "yolov10s"},
#     {"weights": "yolov10n.pt", "name": "yolov10n"}, # just for testing, not sure if it will work
# ]

all are trained on amp, allowed caching

## YOLO26n

In [ ]:
yolo26n = YOLO("yolo26n.pt")

results = yolo26n.train(
    data=f"{dataset.location}/data.yaml",
    epochs=15,
    imgsz=640,
    batch=16,
    workers=2,
    cache=False,
    amp=True,
    patience=5,
    exist_ok=True,                      # Overwrite rather than incrementing to n2, n3
    name="vietnam_plate_lpr_yolo26n"
)

best = YOLO("runs/detect/vietnam_plate_lpr_yolo26n/weights/best.pt")
best.export(format="onnx", imgsz=640, simplify=True)

files.download("runs/detect/vietnam_plate_lpr_yolo26n/weights/best.onnx")

Ultralytics 8.4.176 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Vietnamese-Car-License-Plate--1/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=vietnam_plate_lpr_yolo26n-3, nbs=64, nm

FileNotFoundError: [Errno 2] No such file or directory: 'runs/detect/vietnam_plate_lpr_yolo26n/weights/best.pt'

In [10]:
import glob
print("Found weights:")
print(glob.glob("runs/detect/*/weights/*.pt"))

Found weights:
['runs/detect/vietnam_plate_lpr_yolo26n-3/weights/best.pt', 'runs/detect/vietnam_plate_lpr_yolo26n-3/weights/last.pt']


In [ ]:
best = YOLO("runs/detect/vietnam_plate_lpr_yolo26n-3/weights/best.pt")
best.export(format="onnx", imgsz=640, simplify=True)
files.download("runs/detect/vietnam_plate_lpr_yolo26n-3/weights/best.onnx")

Ultralytics 8.4.176 🚀 Python-3.13.15 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO26n summary (fused): 121 layers, 2,389,735 parameters, 0 gradients, 5.4 GFLOPs

PyTorch: starting from 'runs/detect/vietnam_plate_lpr_yolo26n-3/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (5.2 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 12 packages in 365ms
Prepared 4 packages in 537ms
Installed 4 packages in 35ms
 + colorama==0.4.6
 + onnx==1.23.2
 + onnxruntime==1.31.0
 + onnxslim==0.1.98

requirements: AutoUpdate success ✅ 1.8s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.23.2 opset 18...
ONNX:

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## yolo26s

In [12]:
yolo26s = YOLO("yolo26s.pt")

yolo26s.train(
    data=f"{dataset.location}/data.yaml",
    epochs=15,
    imgsz=640,
    batch=16,
    patience=5,
    amp=True,
    workers=2,

    name="vietnam_plate_lpr_yolo26s"
)

best = YOLO("runs/detect/vietnam_plate_lpr_yolo26s/weights/best.pt")
best.export(format="onnx", imgsz=640, simplify=True)
files.download("runs/detect/vietnam_plate_lpr_yolo26s/weights/best.onnx")

Ultralytics 8.4.176 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Vietnamese-Car-License-Plate--1/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=vietnam_plate_lpr_yolo26s, nbs=64, nms=

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## yolo11n (performance wise)

In [13]:
yolo11n = YOLO("yolo11n.pt")

yolo11n.train(
    data=f"{dataset.location}/data.yaml",
    epochs=15,
    imgsz=640,
    batch=16,
    patience=5,
    amp=True,
    workers=2,

    name="vietnam_plate_lpr_yolo11n"
)

best = YOLO("runs/detect/vietnam_plate_lpr_yolo11n/weights/best.pt")
best.export(format="onnx", imgsz=640, simplify=True)

files.download("runs/detect/vietnam_plate_lpr_yolo11n/weights/best.onnx")

Ultralytics 8.4.176 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Vietnamese-Car-License-Plate--1/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=vietnam_plate_lpr_yolo11n, nbs=64, nms=

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

216pm ended here

## yolo11s

In [ ]:
yolo11s = YOLO("yolo11s.pt")

yolo11s.train(
    data=f"{dataset.location}/data.yaml",
    epochs=15,
    imgsz=640,
    batch=16,
    patience=5,
    amp=True,
    workers=2,

    name="vietnam_plate_lpr_yolo11s"
)

best = YOLO("runs/detect/vietnam_plate_lpr_yolo11s/weights/best.pt")
best.export(format="onnx", imgsz=640, simplify=True)

files.download("runs/detect/vietnam_plate_lpr_yolo11s/weights/best.onnx")

Ultralytics 8.4.176 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)


## yolov10n

In [ ]:
yolov10n = YOLO("yolov10n.pt")

yolov10n.train(
    data=f"{dataset.location}/data.yaml",
    epochs=15,
    imgsz=640,
    batch=16,
    patience=5,
    amp=True,
    workers=2,

    name="vietnam_plate_lpr_yolov10n"
)

best = YOLO("runs/detect/vietnam_plate_lpr_yolov10n/weights/best.pt")
best.export(format="onnx", imgsz=640, simplify=True)

files.download("runs/detect/vietnam_plate_lpr_yolov10n/weights/best.onnx")

## yolov10s

In [ ]:
yolov10s = YOLO("yolov10s.pt")

yolov10s.train(
    data=f"{dataset.location}/data.yaml",
    epochs=15,
    imgsz=640,
    batch=16,
    patience=5,
    amp=True,
    workers=2,

    name="vietnam_plate_lpr_yolov10s"
)

best = YOLO("runs/detect/vietnam_plate_lpr_yolov10s/weights/best.pt")
best.export(format="onnx", imgsz=640, simplify=True)

files.download("runs/detect/vietnam_plate_lpr_yolov10s/weights/best.onnx")

also, on the side note: mp950-95 shows how tight the bounding boxes are fit around the license plate. since we're doing OCR within a bounding box, i'd need to manually expand the bounding boxes before pushing it to an OCR model that will extract the text.

# Download Trained best.pt Weights to Your Local Machine

Model is downloaded to colab notebook's local runtime; please re-download best.pt from there and replace it in your directory's Models_yolo, if retrained.


If on VSCODE's colab extension .ipynb (what I used), go to the colab extension, click on the existing remote server, and find /runs/... to download the .pt model.